# Rename test + ground truth
Jalankan semua sel dari folder dataset, atau ubah `TEST_DIR` dan `CSV_PATH`.

File `test/BISINDO_Kamu_001.mp4` mendapat nama `test_NNN.mp4`. Nomor global mulai 001 berdasarkan urutan nama asli; ekstensi tetap. Label diambil dari nama asli dan disamakan dengan nama kelas train (`apa_kabar`, `kamu`, dst.). Folder train tidak disentuh.

CSV `test_ground_truth.csv` berisi `filename,ground_truth,original_filename`; `filename` tanpa ekstensi, contoh `test_151,kamu,BISINDO_Kamu_001.mp4`. Nama asli disimpan untuk pelacakan. CSV dibuat sebelum rename. Simpan CSV ini; jika proses terhenti, jalankan ulang untuk melanjutkan. CSV yang sudah ada digunakan kembali, tidak ditimpa. Jangan edit dataset/CSV atau menjalankan dua proses rename bersamaan.


In [1]:
from pathlib import Path
import csv
import re

TEST_DIR = Path.cwd() / "test"
CSV_PATH = TEST_DIR.parent / "test_ground_truth.csv"
CLASSES = {"apa_kabar", "bingung", "halo", "kamu", "marah", "saya", "senang", "tidur"}


def rename_test(test_dir, csv_path):
    test_dir, csv_path = Path(test_dir), Path(csv_path)
    files = sorted((p for p in test_dir.iterdir() if p.is_file()), key=lambda p: p.name)
    if not files:
        raise ValueError("Folder test kosong.")
    fields = ["filename", "ground_truth", "original_filename"]
    if csv_path.exists():
        with csv_path.open(newline="", encoding="utf-8") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != fields:
                raise ValueError("Format CSV tidak sesuai; CSV tidak ditimpa.")
            rows = list(reader)
    else:
        rows = []
        for index, source in enumerate(files, 1):
            match = re.fullmatch(r"BISINDO_(.+)_(\d+)", source.stem)
            label = match[1].lower().replace(" ", "_") if match else None
            if label not in CLASSES:
                raise ValueError(f"Label tidak dikenali: {source.name}")
            rows.append(dict(filename=f"test_{index:03d}", ground_truth=label,
                             original_filename=source.name))

    plan = []
    for index, row in enumerate(rows, 1):
        original = row["original_filename"]
        if not original or Path(original).name != original or "/" in original or "\\" in original:
            raise ValueError("Nama file CSV tidak valid.")
        match = re.fullmatch(r"BISINDO_(.+)_(\d+)", Path(original).stem)
        label = match[1].lower().replace(" ", "_") if match else None
        if (row["filename"] != f"test_{index:03d}" or label not in CLASSES
                or row["ground_truth"] != label):
            raise ValueError("Mapping CSV tidak valid.")
        source = test_dir / original
        target = test_dir / (row["filename"] + source.suffix)
        if source.exists() == target.exists():
            raise ValueError(f"File hilang atau nama bentrok: {source.name} / {target.name}")
        plan.append((source, target))
    expected = {source if source.exists() else target for source, target in plan}
    if len(expected) != len(rows) or set(files) != expected:
        raise ValueError("Isi test tidak cocok dengan mapping CSV.")

    # Simpan label sebelum rename; jalankan ulang untuk melanjutkan jika terhenti.
    if not csv_path.exists():
        with csv_path.open("x", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
            stream.flush()
            import os
            os.fsync(stream.fileno())
    changed = 0
    for source, target in plan:
        if source.exists():
            source.rename(target)
            changed += 1
    return changed, len(rows)


In [2]:
changed, total = rename_test(TEST_DIR, CSV_PATH)
print(f"Selesai: {changed} file diubah; {total} label tercatat.")
print(f"CSV: {CSV_PATH.resolve()}")
with CSV_PATH.open(newline="", encoding="utf-8") as stream:
    for row in list(csv.DictReader(stream))[:5]:
        print(row)


Selesai: 400 file diubah; 400 label tercatat.
CSV: C:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\test_ground_truth.csv
{'filename': 'test_001', 'ground_truth': 'apa_kabar', 'original_filename': 'BISINDO_Apa Kabar_001.mp4'}
{'filename': 'test_002', 'ground_truth': 'apa_kabar', 'original_filename': 'BISINDO_Apa Kabar_002.mp4'}
{'filename': 'test_003', 'ground_truth': 'apa_kabar', 'original_filename': 'BISINDO_Apa Kabar_003.mp4'}
{'filename': 'test_004', 'ground_truth': 'apa_kabar', 'original_filename': 'BISINDO_Apa Kabar_004.mp4'}
{'filename': 'test_005', 'ground_truth': 'apa_kabar', 'original_filename': 'BISINDO_Apa Kabar_005.mp4'}
